> **Superseded 18 Sep 2026.** The debate now runs on a Mac (LM Studio, Qwen3.8) and this workstation (llama.cpp, InternVL), each judge reading its own dataset copy and addressed by Q_ID only -- see `VQA/agent.debate.mc.py` and `VQA/debate.rank.mc.py`. This notebook describes the earlier Colab/Drive design and is kept for reference.


# Agent B - InternVL3.5-30B-A3B Q4 on llama.cpp

The other side of the debate. Same contract: this session reads frames itself
from `/content/frames448`, which `--media-path` exposes to `file://` URLs.

A 24GB GPU is enough - 18.6 GB of weights plus a 0.6 GB vision projector.

Run top to bottom, then paste the printed flags into the orchestrator.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# Prebuilt CUDA binaries, so no 20-minute source build.
!wget -q -O /content/llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b10976/llama-b10976-bin-ubuntu-cuda-12.8-x64.tar.gz
!mkdir -p /content/llama && tar -xzf /content/llama.tar.gz -C /content/llama --strip-components=1
!ls /content/llama/bin | head -5
!pip -q install huggingface_hub


In [ ]:
from huggingface_hub import hf_hub_download
REPO = "bartowski/OpenGVLab_InternVL3_5-30B-A3B-GGUF"
weights = hf_hub_download(REPO, "OpenGVLab_InternVL3_5-30B-A3B-Q4_K_M.gguf",
                          local_dir="/content/gguf")
mmproj = hf_hub_download(REPO, "mmproj-OpenGVLab_InternVL3_5-30B-A3B-f16.gguf",
                         local_dir="/content/gguf")
print(weights, mmproj)


In [ ]:
#@title Session settings
DRIVE_ROOT = "/content/drive/MyDrive/TREC_2026"  #@param {type:"string"}
import secrets
API_KEY = secrets.token_urlsafe(16)
from google.colab import drive
drive.mount("/content/drive")
print("API key for this session:", API_KEY)


In [ ]:
# Cut the debate frames from the videos already on Drive.
# 448x448, mid-point sampling -- the convention VQA/extract.frames.py uses, and
# the same square resize InternVL's own video path does, so the agents see what
# the panel saw, only more of it. ~43k frames, roughly 20-40 min. Resumable.
import json, subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE  = Path(DRIVE_ROOT)
VIDEOS = DRIVE / "videos_TEST_VQA" / "videos"
OUT    = Path("/content/frames448")
manifest = json.load(open(DRIVE / "debate.manifest.json"))["videos"]
print(len(manifest), "videos,", sum(v["n_frames"] for v in manifest.values()), "frames")

def source(stem):
    for name in (stem + ".mp4", stem + ".webm"):
        if (VIDEOS / name).exists():
            return VIDEOS / name
    hits = sorted(VIDEOS.glob(stem + ".*"))
    return hits[0] if hits else None

def cut(item):
    stem, meta = item
    n, duration = int(meta["n_frames"]), float(meta["duration_sec"])
    out = OUT / stem
    if len(list(out.glob("*.jpg"))) == n:          # already cut -> resumable
        return "skip"
    src = source(stem)
    if src is None:
        return "missing"
    out.mkdir(parents=True, exist_ok=True)
    offset = duration / (2 * n)
    vf = "fps=%d/%s:start_time=%.4f,scale=448:448" % (n, duration, offset)
    cmd = ["ffmpeg", "-v", "error", "-an", "-sn", "-i", str(src), "-vf", vf,
           "-q:v", "3", "-frames:v", str(n), str(out / "%03d.jpg"), "-y"]
    subprocess.run(cmd, capture_output=True)
    got = len(list(out.glob("*.jpg")))
    return "ok" if got == n else "short"

with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(cut, manifest.items()))

print(Counter(results))
print("under-cut or missing videos stay at the baseline in the run file")


In [ ]:
import subprocess, time, requests
PORT, MODEL = 8013, "internvl3.5-30b-a3b"

# --media-path is what lets the orchestrator send file:// references instead of
# ~800 KB of base64 per call. The frames are already 448x448, i.e. one InternVL
# tile each -- llama.cpp otherwise slices a portrait frame into about five tiles
# (~1,280 tokens) and 32 of those overflow the context, which is exactly how
# InternVL's first run broke.
cmd = (
    "/content/llama/bin/llama-server -m " + weights + " --mmproj " + mmproj +
    " --jinja -ngl all -fa on -c 32768 -np 2" +
    " --cache-type-k q8_0 --cache-type-v q8_0" +
    " --media-path /content/frames448 --api-key " + API_KEY +
    " --alias " + MODEL + " --host 0.0.0.0 --port " + str(PORT) + " --no-webui"
)
log = open("/content/llama.log", "w")
subprocess.Popen(cmd, shell=True, stdout=log, stderr=log)

for _ in range(90):
    try:
        if requests.get("http://localhost:%d/health" % PORT, timeout=5).ok:
            print("server up")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    print(open("/content/llama.log").read()[-3000:])


In [ ]:
# A server started without --mmproj accepts image parts, silently drops them,
# and answers from the text alone. Check before trusting a single ranking.
import requests
print(requests.get("http://localhost:%d/props" % PORT, timeout=30).json().get("modalities"))

# The FIRST real request is where a broken kernel build shows up -- as an
# engine death, after /v1/models has already answered 200. Find it here, not 300
# queries into the run.
import glob, requests
frame = sorted(glob.glob("/content/frames448/*/001.jpg"))[0]
body = {"model": MODEL, "max_tokens": 32, "messages": [{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": "file://" + frame}},
    {"type": "text", "text": "Reply with one word: what is in this frame?"}]}]}
r = requests.post("http://localhost:%d/v1/chat/completions" % PORT,
                  headers={"Authorization": "Bearer " + API_KEY},
                  json=body, timeout=300)
print(r.status_code, r.text[:300])


In [ ]:
SIDE = "b"
# A public URL for this server. Paste the three flags into the orchestrator on
# the workstation. Quick tunnels are public, which is why the server has a key.
import re, subprocess
subprocess.run("wget -q -O /content/cloudflared "
               "https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True)
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                         "--url", "http://localhost:%d" % PORT],
                        stderr=subprocess.PIPE, text=True)
url = None
for line in proc.stderr:
    hit = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if hit:
        url = hit.group(0)
        break

print("\n" + "=" * 68)
print("  --host-%s %s" % (SIDE, url))
print("  --model-%s %s" % (SIDE, MODEL))
print("  --api-key-%s %s" % (SIDE, API_KEY))
print("=" * 68)
print("Leave this cell running -- closing it drops the tunnel.")
